# LLM Colors Preference Analysis
Analyze the results of adaptive preference elicitation from LLMs on the colors domain.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Set project root to import grums modules if needed
ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", palette="muted")

## 1. Load Data
Set the `EXP_DIR` to your experiment output folder.

In [ ]:
# TODO: SET your actual experiment directory here (relative to project root)
EXP_DIR = ROOT / "results" / "llm" / "llm_colors_20260325-163229"
output_dir = EXP_DIR / "outputs"

if not EXP_DIR.exists():
    print(f"WARNING: Directory {EXP_DIR} not found.")

results = []
for f in sorted(output_dir.glob("*.json")):
    with open(f, "r") as j:
        results.append(json.load(j))

if results:
    print(f"Loaded {len(results)} trials from {EXP_DIR.name} ({results[0]['model_id']})")
else:
    print("No results found!")

## 2. Global Preference Rankings ($\delta$)
Which colors does the model prefer overall?

In [ ]:
color_names = ["blue", "red", "green", "purple", "yellow"]

all_deltas = []
for r in results:
    last_step = sorted(map(int, r["history"].keys()))[-1]
    delta = r["history"][str(last_step)]["delta"]
    all_deltas.append(delta)

if all_deltas:
    df_delta = pd.DataFrame(all_deltas, columns=color_names)
    
    # Pre-melting for Seaborn (tidier data)
    df_melt = df_delta.melt(var_name="Color", value_name="Delta")
    
    # Standardizing ranking order
    order = df_melt.groupby("Color")["Delta"].mean().sort_values(ascending=False).index

    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_melt, x="Color", y="Delta", order=order, errorbar="sd")
    plt.title(f"Elicited Preference Scores ($\delta$) - Averaged across Seeds")
    plt.ylabel("Preference Score")
    plt.xlabel("Alternative (Color)")
    plt.show()

## 3. Interaction Matrix ($B$)
How do different PCA features (prompt styles) modulate the preference?

In [ ]:
all_B = []
for r in results:
    last_step = sorted(map(int, r["history"].keys()))[-1]
    B = np.array(r["history"][str(last_step)]["interaction"])
    all_B.append(B)

if all_B:
    mean_B = np.mean(all_B, axis=0)

    plt.figure(figsize=(12, 8))
    sns.heatmap(mean_B, annot=True, cmap="coolwarm", xticklabels=color_names, yticklabels=[f"PCA_{i}" for i in range(mean_B.shape[0])])
    plt.title("Mean Interaction Matrix (B) - PCA Features vs Colors")
    plt.xlabel("Colors")
    plt.ylabel("PCA Dimension")
    plt.show()

## 4. Convergence over Steps
Track how the estimated parameters stabilize.

In [ ]:
plt.figure(figsize=(12, 6))

if results:
    for i, r in enumerate(results[:3]): # Plot first 3 trials
        steps = sorted(map(int, r["history"].keys()))
        deltas = [r["history"][str(s)]["delta"] for s in steps]
        deltas = np.array(deltas)
        
        for j in range(len(color_names)):
            ls = "-" if i == 0 else "--" if i == 1 else ":"
            plt.plot(steps, deltas[:, j], alpha=0.5, linestyle=ls, 
                     label=f"{color_names[j]} (trial {i})" if j == 0 or i == 0 else "")

plt.title("Stabilization of $\delta$ over Elicitation Steps (Top 3 Trials)")
plt.xlabel("Observation Step")
plt.ylabel("Estimated Delta")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()